# Moment-Curvature Analysis of a RC Column Fiber Section

This notebook demonstrates how to use **Gmsh** to mesh a reinforced concrete (RC)
rectangular column cross-section into fibers, then perform a **moment-curvature analysis**
using **OpenSeesPy**.

We show two approaches:
1. **Raw Gmsh + OpenSeesPy** — direct use of the Gmsh Python API
2. **pyGmsh + OpenSeesPy** — using the pyGmsh wrapper library

## Column Section Properties

| Property | Value |
|---|---|
| Width (b) | 400 mm |
| Depth (h) | 600 mm |
| Cover | 40 mm |
| Concrete f'c | 30 MPa |
| Steel fy | 420 MPa |
| Steel Es | 200,000 MPa |
| Top bars | 3 ø 25 mm |
| Bottom bars | 3 ø 25 mm |
| Side bars | 2 ø 16 mm (each side) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Section geometry ──
b      = 400.0    # width  [mm]
h      = 600.0    # depth  [mm]
cover  = 40.0     # clear cover [mm]

# ── Reinforcement ──
db_main = 25.0    # main bar diameter [mm]
db_side = 16.0    # side bar diameter [mm]
n_top   = 3       # number of top bars
n_bot   = 3       # number of bottom bars
n_side  = 2       # number of side bars per side

As_main = np.pi * db_main**2 / 4  # area per main bar
As_side = np.pi * db_side**2 / 4  # area per side bar

# ── Material properties ──
fc    = 30.0       # concrete compressive strength [MPa]
fy    = 420.0      # steel yield strength [MPa]
Es    = 200000.0   # steel elastic modulus [MPa]

# ── Derived concrete parameters (Mander confined model) ──
Ec    = 4700.0 * np.sqrt(fc)   # concrete elastic modulus [MPa]
eps_c0 = 2 * fc / Ec            # strain at peak stress (unconfined)
eps_cu = 0.005                   # ultimate strain (confined, approximate)
ft     = 0.62 * np.sqrt(fc)     # tensile strength [MPa]
eps_tu = ft / Ec                 # tensile cracking strain

print(f"Ec = {Ec:.1f} MPa")
print(f"eps_c0 = {eps_c0:.5f}")
print(f"ft = {ft:.2f} MPa")
print(f"As_main = {As_main:.1f} mm²,  As_side = {As_side:.1f} mm²")

---
## Part 1: Raw Gmsh Approach

We use the Gmsh Python API directly to:
1. Build the rectangular section with core and cover regions
2. Generate a quad-dominant mesh
3. Extract fiber centroids and areas for OpenSees

In [ ]:
import gmsh
import sys

gmsh.initialize()
gmsh.model.add("RC_Section_Raw")

# ── Geometry: Outer rectangle (full section) ──
# Origin at centroid of the section
p1 = gmsh.model.occ.addPoint(-b/2,  -h/2, 0)
p2 = gmsh.model.occ.addPoint( b/2,  -h/2, 0)
p3 = gmsh.model.occ.addPoint( b/2,   h/2, 0)
p4 = gmsh.model.occ.addPoint(-b/2,   h/2, 0)

l1 = gmsh.model.occ.addLine(p1, p2)
l2 = gmsh.model.occ.addLine(p2, p3)
l3 = gmsh.model.occ.addLine(p3, p4)
l4 = gmsh.model.occ.addLine(p4, p1)

outer_loop = gmsh.model.occ.addCurveLoop([l1, l2, l3, l4])

# ── Geometry: Inner rectangle (confined core) ──
ci = cover + db_main / 2  # distance to bar center (stirrup center-line)
p5 = gmsh.model.occ.addPoint(-b/2 + ci, -h/2 + ci, 0)
p6 = gmsh.model.occ.addPoint( b/2 - ci, -h/2 + ci, 0)
p7 = gmsh.model.occ.addPoint( b/2 - ci,  h/2 - ci, 0)
p8 = gmsh.model.occ.addPoint(-b/2 + ci,  h/2 - ci, 0)

l5 = gmsh.model.occ.addLine(p5, p6)
l6 = gmsh.model.occ.addLine(p6, p7)
l7 = gmsh.model.occ.addLine(p7, p8)
l8 = gmsh.model.occ.addLine(p8, p5)

inner_loop = gmsh.model.occ.addCurveLoop([l5, l6, l7, l8])

# ── Create surfaces using boolean fragment ──
# Core = inner rectangle, Cover = outer minus inner
core_surf  = gmsh.model.occ.addPlaneSurface([inner_loop])
outer_surf = gmsh.model.occ.addPlaneSurface([outer_loop, inner_loop])  # cover (with hole)

gmsh.model.occ.synchronize()

# ── Physical groups ──
gmsh.model.addPhysicalGroup(2, [core_surf],  name="Core_Concrete")
gmsh.model.addPhysicalGroup(2, [outer_surf], name="Cover_Concrete")

print("Geometry created: core + cover surfaces")
print(f"  Core inner dimensions: {b - 2*ci:.0f} x {h - 2*ci:.0f} mm")
print(f"  Cover thickness: {ci:.1f} mm")

In [ ]:
# ── Mesh settings ──
# We want a quad-dominant mesh for well-shaped fibers
target_size = 20.0  # approximate fiber size [mm]

gmsh.option.setNumber("Mesh.MeshSizeMin", target_size * 0.8)
gmsh.option.setNumber("Mesh.MeshSizeMax", target_size * 1.2)
gmsh.option.setNumber("Mesh.Algorithm", 8)  # Frontal-Delaunay for quads
gmsh.option.setNumber("Mesh.RecombineAll", 1)  # force quads
gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 2)  # simple full-quad

# Generate 2D mesh
gmsh.model.mesh.generate(2)

# ── Extract mesh data ──
node_tags, node_coords, _ = gmsh.model.mesh.getNodes()
node_coords = node_coords.reshape(-1, 3)

# Build node coordinate map
node_map = {int(t): node_coords[i, :2] for i, t in enumerate(node_tags)}

print(f"Mesh generated: {len(node_tags)} nodes")

In [ ]:
def extract_fibers_from_gmsh(physical_group_name):
    """
    Extract fiber data (centroid y, centroid z, area) from a Gmsh physical group.
    Returns arrays suitable for OpenSees fiber section definition.
    
    Each mesh element (quad or triangle) becomes one fiber with:
    - centroid at the element center
    - area computed via the shoelace formula
    """
    # Get physical group tag by name
    pg_dimtags = gmsh.model.getPhysicalGroups(dim=2)
    pg_tag = None
    for dim, tag in pg_dimtags:
        name = gmsh.model.getPhysicalName(dim, tag)
        if name == physical_group_name:
            pg_tag = tag
            break
    if pg_tag is None:
        raise ValueError(f"Physical group '{physical_group_name}' not found")
    
    # Get entities in this physical group
    entities = gmsh.model.getEntitiesForPhysicalGroup(2, pg_tag)
    
    fibers_y = []
    fibers_z = []
    fibers_A = []
    
    for entity_tag in entities:
        elem_types, elem_tags, elem_node_tags = gmsh.model.mesh.getElements(2, entity_tag)
        
        for etype, etags, enodes in zip(elem_types, elem_tags, elem_node_tags):
            # getElementProperties returns: (name, dim, order, numNodes, paramCoord)
            props = gmsh.model.mesh.getElementProperties(etype)
            npe = props[3]  # numNodes is at index 3 (NOT index 2, which is order)
            enodes = enodes.reshape(-1, npe).astype(int)
            
            for elem_nodes in enodes:
                # Get node coordinates
                coords = np.array([node_map[n] for n in elem_nodes])
                
                # Centroid
                cy = np.mean(coords[:, 0])  # y = horizontal
                cz = np.mean(coords[:, 1])  # z = vertical
                
                # Area using shoelace formula
                n_nodes = len(coords)
                area = 0.0
                for i in range(n_nodes):
                    j = (i + 1) % n_nodes
                    area += coords[i, 0] * coords[j, 1]
                    area -= coords[j, 0] * coords[i, 1]
                area = abs(area) / 2.0
                
                fibers_y.append(cy)
                fibers_z.append(cz)
                fibers_A.append(area)
    
    return np.array(fibers_y), np.array(fibers_z), np.array(fibers_A)


# Extract core and cover fibers
core_y, core_z, core_A = extract_fibers_from_gmsh("Core_Concrete")
cover_y, cover_z, cover_A = extract_fibers_from_gmsh("Cover_Concrete")

print(f"Core fibers:  {len(core_A)},  total area = {core_A.sum():.0f} mm²")
print(f"Cover fibers: {len(cover_A)},  total area = {cover_A.sum():.0f} mm²")
print(f"Total section area: {core_A.sum() + cover_A.sum():.0f} mm²  "
      f"(analytical: {b*h:.0f} mm²)")

gmsh.finalize()

In [ ]:
# ── Rebar positions ──
# Bars are placed at the stirrup center line
rebar_y = []
rebar_z = []
rebar_A = []

# Bottom bars (3 ø 25)
y_bot = np.linspace(-b/2 + ci, b/2 - ci, n_bot)
z_bot = -h/2 + ci
for y in y_bot:
    rebar_y.append(y)
    rebar_z.append(z_bot)
    rebar_A.append(As_main)

# Top bars (3 ø 25)
y_top = np.linspace(-b/2 + ci, b/2 - ci, n_top)
z_top = h/2 - ci
for y in y_top:
    rebar_y.append(y)
    rebar_z.append(z_top)
    rebar_A.append(As_main)

# Side bars (2 ø 16 each side)
z_side = np.linspace(-h/2 + ci, h/2 - ci, n_side + 2)[1:-1]  # interior positions
for z in z_side:
    # Left side
    rebar_y.append(-b/2 + ci)
    rebar_z.append(z)
    rebar_A.append(As_side)
    # Right side
    rebar_y.append(b/2 - ci)
    rebar_z.append(z)
    rebar_A.append(As_side)

rebar_y = np.array(rebar_y)
rebar_z = np.array(rebar_z)
rebar_A = np.array(rebar_A)

print(f"Total rebar: {len(rebar_A)} bars")
print(f"Total steel area: {rebar_A.sum():.0f} mm²")
print(f"Reinforcement ratio: {rebar_A.sum() / (b*h) * 100:.2f}%")

In [ ]:
# ── Visualize the fiber section ──
fig, ax = plt.subplots(1, 1, figsize=(8, 10))

# Cover fibers
sc_cover = ax.scatter(cover_y, cover_z, s=cover_A * 0.03, 
                       c='lightblue', edgecolors='steelblue', 
                       linewidths=0.5, alpha=0.7, label='Cover concrete')

# Core fibers
sc_core = ax.scatter(core_y, core_z, s=core_A * 0.03,
                      c='lightcoral', edgecolors='firebrick',
                      linewidths=0.5, alpha=0.7, label='Core concrete (confined)')

# Rebar
ax.scatter(rebar_y, rebar_z, s=rebar_A * 0.5, 
           c='black', marker='o', zorder=5, label='Steel rebar')

# Section outline
rect_out = plt.Rectangle((-b/2, -h/2), b, h, fill=False, 
                          edgecolor='black', linewidth=2)
rect_core = plt.Rectangle((-b/2+ci, -h/2+ci), b-2*ci, h-2*ci, fill=False,
                           edgecolor='firebrick', linewidth=1.5, linestyle='--')
ax.add_patch(rect_out)
ax.add_patch(rect_core)

ax.set_xlabel('y [mm]')
ax.set_ylabel('z [mm]')
ax.set_title('RC Column Fiber Section (Raw Gmsh Approach)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2: pyGmsh Approach

Now the same section using pyGmsh's OOP wrapper.
The key advantage: cleaner API, physical groups by name, and built-in mesh extraction.

In [ ]:
import sys
sys.path.insert(0, '../src')
from pyGmsh import pyGmsh

g = pyGmsh(model_name="RC_Section_pyGmsh", verbose=True)
g.initialize()

# ── Geometry using pyGmsh Model API ──
# Outer rectangle (full section)
p1 = g.model.add_point(-b/2, -h/2, 0)
p2 = g.model.add_point( b/2, -h/2, 0)
p3 = g.model.add_point( b/2,  h/2, 0)
p4 = g.model.add_point(-b/2,  h/2, 0)

l1 = g.model.add_line(p1, p2)
l2 = g.model.add_line(p2, p3)
l3 = g.model.add_line(p3, p4)
l4 = g.model.add_line(p4, p1)

outer_loop = g.model.add_curve_loop([l1, l2, l3, l4])

# Inner rectangle (confined core)
p5 = g.model.add_point(-b/2 + ci, -h/2 + ci, 0)
p6 = g.model.add_point( b/2 - ci, -h/2 + ci, 0)
p7 = g.model.add_point( b/2 - ci,  h/2 - ci, 0)
p8 = g.model.add_point(-b/2 + ci,  h/2 - ci, 0)

l5 = g.model.add_line(p5, p6)
l6 = g.model.add_line(p6, p7)
l7 = g.model.add_line(p7, p8)
l8 = g.model.add_line(p8, p5)

inner_loop = g.model.add_curve_loop([l5, l6, l7, l8])

# Surfaces
# add_plane_surface accepts [outer_loop, hole1, ...] — first is boundary, rest are holes
core_surf  = g.model.add_plane_surface(inner_loop)
cover_surf = g.model.add_plane_surface([outer_loop, inner_loop])

g.model.sync()

# ── Physical groups via pyGmsh ──
g.physical.add(2, [core_surf],  name="Core_Concrete")
g.physical.add(2, [cover_surf], name="Cover_Concrete")

# ── Mesh settings (all through pyGmsh API) ──
g.mesh.set_size_global(min_size=target_size * 0.8, max_size=target_size * 1.2)

# Per-surface: set algorithm and recombine for quad mesh
for surf in [core_surf, cover_surf]:
    g.mesh.set_algorithm(surf, algorithm=8)   # Frontal-Delaunay for quads
    g.mesh.set_recombine(surf)                 # force quad recombination

g.mesh.generate(dim=2)

print("pyGmsh mesh generated successfully!")

In [ ]:
# ── Extract fiber data using pyGmsh + raw gmsh for element access ──
# Note: g.mesh.get_fem_data() assumes uniform element types. For quad-dominant
# meshes that may contain a few triangles, we extract per physical group directly.

import gmsh as _gmsh2

# Build node map from pyGmsh's mesh
nodes = g.mesh.get_nodes()
pg_node_tags   = nodes['tags']
pg_node_coords = nodes['coords']
pg_node_map = {int(t): pg_node_coords[i, :2] for i, t in enumerate(pg_node_tags)}

def extract_fibers_pyGmsh(physical_group_name, nmap):
    """
    Extract fiber data from a physical group using pyGmsh's active Gmsh session.
    Each 2D element (triangle or quad) becomes one fiber.
    """
    pg_dimtags = _gmsh2.model.getPhysicalGroups(dim=2)
    pg_tag = None
    for dim, tag in pg_dimtags:
        name = _gmsh2.model.getPhysicalName(dim, tag)
        if name == physical_group_name:
            pg_tag = tag
            break
    
    entities = _gmsh2.model.getEntitiesForPhysicalGroup(2, pg_tag)
    
    fibers_y, fibers_z, fibers_A = [], [], []
    
    for entity_tag in entities:
        elem_types, elem_tags, elem_node_tags = _gmsh2.model.mesh.getElements(2, entity_tag)
        for etype, etags, enodes in zip(elem_types, elem_tags, elem_node_tags):
            # getElementProperties returns: (name, dim, order, numNodes, paramCoord)
            props = _gmsh2.model.mesh.getElementProperties(etype)
            npe = props[3]  # numNodes at index 3
            enodes = enodes.reshape(-1, npe).astype(int)
            
            for elem_nodes in enodes:
                coords = np.array([nmap[n] for n in elem_nodes])
                cy = np.mean(coords[:, 0])
                cz = np.mean(coords[:, 1])
                # Shoelace formula for polygon area
                nn = len(coords)
                area = abs(sum(
                    coords[i, 0] * coords[(i+1)%nn, 1] - coords[(i+1)%nn, 0] * coords[i, 1]
                    for i in range(nn)
                )) / 2.0
                fibers_y.append(cy)
                fibers_z.append(cz)
                fibers_A.append(area)
    
    return np.array(fibers_y), np.array(fibers_z), np.array(fibers_A)


pg_core_y,  pg_core_z,  pg_core_A  = extract_fibers_pyGmsh("Core_Concrete",  pg_node_map)
pg_cover_y, pg_cover_z, pg_cover_A = extract_fibers_pyGmsh("Cover_Concrete", pg_node_map)

print(f"pyGmsh — Core fibers:  {len(pg_core_A)},  area = {pg_core_A.sum():.0f} mm²")
print(f"pyGmsh — Cover fibers: {len(pg_cover_A)},  area = {pg_cover_A.sum():.0f} mm²")
print(f"pyGmsh — Total area:   {pg_core_A.sum() + pg_cover_A.sum():.0f} mm²")

g.finalize()

---
## Part 3: Moment-Curvature Analysis with OpenSeesPy

We use the fiber data from the raw Gmsh approach (Part 1) to build an OpenSees
fiber section and perform moment-curvature analysis.

### Strategy
- Single zero-length element with a fiber section
- Node 1 is fixed, Node 2 has a constant axial load and imposed rotation
- We apply incremental curvature and record moment

In [ ]:
import openseespy.opensees as ops

def run_moment_curvature(core_y, core_z, core_A,
                          cover_y, cover_z, cover_A,
                          rebar_y, rebar_z, rebar_A,
                          P_axial=0.0,
                          max_curvature=0.0001,
                          n_steps=200,
                          label=""):
    """
    Run a moment-curvature analysis using OpenSeesPy with fiber-by-fiber
    section definition from Gmsh mesh data.
    
    Parameters
    ----------
    core_y, core_z, core_A : fiber data for confined core concrete
    cover_y, cover_z, cover_A : fiber data for unconfined cover concrete
    rebar_y, rebar_z, rebar_A : steel rebar fiber data
    P_axial : axial load [N] (negative = compression)
    max_curvature : target max curvature [1/mm]
    n_steps : number of analysis steps
    label : label for this run
    
    Returns
    -------
    curvature, moment : arrays
    """
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    
    # ── Materials ──
    # Tag 1: Confined concrete (core) — Concrete02
    #   Concrete02 $tag $fpc $epsc0 $fpcu $epsU $lambda $ft $Ets
    fpc_conf  = -1.3 * fc        # confined peak stress (30% increase)
    epsc0_conf = -0.004           # confined strain at peak
    fpcu_conf  = -0.2 * fc        # residual strength
    epsU_conf  = -0.014           # confined ultimate strain
    lam        = 0.1              # ratio between unloading slope at epsU and initial slope
    
    ops.uniaxialMaterial('Concrete02', 1,
                          fpc_conf, epsc0_conf, fpcu_conf, epsU_conf,
                          lam, ft, ft / 0.002)
    
    # Tag 2: Unconfined concrete (cover) — Concrete02
    fpc_unconf  = -fc
    epsc0_unconf = -eps_c0
    fpcu_unconf  = 0.0            # cover spalls completely
    epsU_unconf  = -0.006
    
    ops.uniaxialMaterial('Concrete02', 2,
                          fpc_unconf, epsc0_unconf, fpcu_unconf, epsU_unconf,
                          lam, ft, ft / 0.002)
    
    # Tag 3: Reinforcing steel — Steel02
    #   Steel02 $tag $Fy $E $b $R0 $cR1 $cR2
    ops.uniaxialMaterial('Steel02', 3,
                          fy, Es, 0.01,
                          18.5, 0.925, 0.15)
    
    # ── Fiber Section ──
    sec_tag = 1
    ops.section('Fiber', sec_tag)
    
    # Add core concrete fibers (confined, material 1)
    for y, z, A in zip(core_y, core_z, core_A):
        ops.fiber(float(z), float(y), float(A), 1)  # fiber(z, y, A, matTag)
    
    # Add cover concrete fibers (unconfined, material 2)
    for y, z, A in zip(cover_y, cover_z, cover_A):
        ops.fiber(float(z), float(y), float(A), 2)
    
    # Add steel rebar fibers (material 3)
    for y, z, A in zip(rebar_y, rebar_z, rebar_A):
        ops.fiber(float(z), float(y), float(A), 3)
    
    n_fibers = len(core_A) + len(cover_A) + len(rebar_A)
    print(f"[{label}] Section built with {n_fibers} fibers "
          f"({len(core_A)} core, {len(cover_A)} cover, {len(rebar_A)} steel)")
    
    # ── Zero-length element model ──
    ops.node(1, 0.0, 0.0)
    ops.node(2, 0.0, 0.0)
    
    ops.fix(1, 1, 1, 1)
    ops.fix(2, 0, 1, 0)  # free in axial and rotation, fixed in transverse
    
    ops.element('zeroLengthSection', 1, 1, 2, sec_tag)
    
    # ── Apply constant axial load ──
    if abs(P_axial) > 0:
        ops.timeSeries('Constant', 1)
        ops.pattern('Plain', 1, 1)
        ops.load(2, P_axial, 0.0, 0.0)
        
        # Solve for axial load
        ops.constraints('Plain')
        ops.numberer('Plain')
        ops.system('BandGeneral')
        ops.test('NormUnbalance', 1.0e-6, 10)
        ops.algorithm('Newton')
        ops.integrator('LoadControl', 1.0)
        ops.analysis('Static')
        ops.analyze(1)
        ops.loadConst('-time', 0.0)
    
    # ── Apply curvature via displacement control on rotation DOF ──
    ops.timeSeries('Linear', 2)
    ops.pattern('Plain', 2, 2)
    ops.load(2, 0.0, 0.0, 1.0)  # unit moment reference load
    
    d_curvature = max_curvature / n_steps
    
    ops.constraints('Plain')
    ops.numberer('Plain')
    ops.system('BandGeneral')
    ops.test('NormUnbalance', 1.0e-6, 20)
    ops.algorithm('Newton')
    ops.integrator('DisplacementControl', 2, 3, d_curvature)
    ops.analysis('Static')
    
    # ── Run analysis and record ──
    curvature = [0.0]
    moment = [0.0]
    
    for i in range(n_steps):
        ok = ops.analyze(1)
        if ok != 0:
            ops.algorithm('NewtonLineSearch', 0.8)
            ok = ops.analyze(1)
            ops.algorithm('Newton')
        if ok != 0:
            print(f"  Analysis failed at step {i+1}/{n_steps}")
            break
        
        curvature.append(ops.nodeDisp(2, 3))   # rotation = curvature
        moment.append(ops.eleForce(1, 3))       # moment at element
    
    print(f"  Completed {len(curvature)-1}/{n_steps} steps")
    
    ops.wipe()
    return np.array(curvature), np.abs(np.array(moment))

In [ ]:
# ── Run moment-curvature: Raw Gmsh fibers ──
P_axial = -0.1 * fc * b * h  # 10% axial load ratio (compression)
print(f"Axial load P = {P_axial/1000:.0f} kN ({P_axial/(fc*b*h)*100:.0f}% of Ag*f'c)")

curv_raw, moment_raw = run_moment_curvature(
    core_y, core_z, core_A,
    cover_y, cover_z, cover_A,
    rebar_y, rebar_z, rebar_A,
    P_axial=P_axial,
    max_curvature=8e-5,
    n_steps=300,
    label="Raw Gmsh"
)

In [ ]:
# ── Run moment-curvature: pyGmsh fibers ──
curv_pg, moment_pg = run_moment_curvature(
    pg_core_y, pg_core_z, pg_core_A,
    pg_cover_y, pg_cover_z, pg_cover_A,
    rebar_y, rebar_z, rebar_A,
    P_axial=P_axial,
    max_curvature=8e-5,
    n_steps=300,
    label="pyGmsh"
)

---
## Part 4: Results and Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Moment-Curvature curves ──
ax = axes[0]
ax.plot(curv_raw * 1e6, moment_raw / 1e6, 'b-', linewidth=2, label='Raw Gmsh')
ax.plot(curv_pg  * 1e6, moment_pg  / 1e6, 'r--', linewidth=2, label='pyGmsh')
ax.set_xlabel('Curvature [×10⁻⁶ 1/mm]')
ax.set_ylabel('Moment [kN·m]')
ax.set_title('Moment-Curvature Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

# ── Fiber section view (side-by-side check) ──
ax2 = axes[1]
ax2.scatter(core_y, core_z, s=5, c='lightcoral', alpha=0.6, label=f'Core ({len(core_A)} fibers)')
ax2.scatter(cover_y, cover_z, s=5, c='lightblue', alpha=0.6, label=f'Cover ({len(cover_A)} fibers)')
ax2.scatter(rebar_y, rebar_z, s=40, c='black', marker='o', zorder=5, label=f'Steel ({len(rebar_A)} bars)')
rect_out = plt.Rectangle((-b/2, -h/2), b, h, fill=False, edgecolor='black', linewidth=2)
rect_core = plt.Rectangle((-b/2+ci, -h/2+ci), b-2*ci, h-2*ci, fill=False,
                           edgecolor='firebrick', linewidth=1.5, linestyle='--')
ax2.add_patch(rect_out)
ax2.add_patch(rect_core)
ax2.set_xlabel('y [mm]')
ax2.set_ylabel('z [mm]')
ax2.set_title('Fiber Section Layout')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Summary ──
print("\n" + "="*60)
print("MOMENT-CURVATURE SUMMARY")
print("="*60)
print(f"Section: {b:.0f} x {h:.0f} mm")
print(f"Axial load: {P_axial/1000:.0f} kN")
print(f"\nRaw Gmsh approach:")
print(f"  Peak moment: {moment_raw.max()/1e6:.1f} kN·m")
print(f"  At curvature: {curv_raw[np.argmax(moment_raw)]*1e6:.2f} ×10⁻⁶ 1/mm")
print(f"\npyGmsh approach:")
print(f"  Peak moment: {moment_pg.max()/1e6:.1f} kN·m")
print(f"  At curvature: {curv_pg[np.argmax(moment_pg)]*1e6:.2f} ×10⁻⁶ 1/mm")
print(f"\nDifference in peak moment: {abs(moment_raw.max() - moment_pg.max())/moment_raw.max()*100:.2f}%")

---
## Part 5: Using the `RectangularColumnSection` Class

All the meshing, fiber extraction, material definition, and OpenSees injection from
Parts 1-3 is now encapsulated in a single class. Three lines of code replace ~100.

In [ ]:
from sections import RectangularColumnSection

# ── Define the section in one shot ──
sec = RectangularColumnSection(
    b=400, h=600, cover=40,
    top_bars=(3, 25), bot_bars=(3, 25),
    side_bars=(2, 16),
    fc=30, fy=420, Es=200_000,
    confinement_factor=1.3,
    mesh_size=20,
)

print(sec.summary())

In [ ]:
# ── Visualise the section ──
sec.plot(show=True)

In [ ]:
# ── Run moment-curvature with the class ──
ops.wipe()
ops.model('basic', '-ndm', 2, '-ndf', 3)

# Build the section (defines materials + fiber section in one call)
mat_info = sec.build(sec_tag=1, start_mat_tag=10)
print("Materials created:", mat_info)

# ── Zero-length element model (same as before) ──
ops.node(1, 0.0, 0.0)
ops.node(2, 0.0, 0.0)
ops.fix(1, 1, 1, 1)
ops.fix(2, 0, 1, 0)
ops.element('zeroLengthSection', 1, 1, 2, 1)

# ── Constant axial load ──
P_axial_class = -0.1 * fc * b * h
ops.timeSeries('Constant', 1)
ops.pattern('Plain', 1, 1)
ops.load(2, P_axial_class, 0.0, 0.0)

ops.constraints('Plain')
ops.numberer('Plain')
ops.system('BandGeneral')
ops.test('NormUnbalance', 1.0e-6, 10)
ops.algorithm('Newton')
ops.integrator('LoadControl', 1.0)
ops.analysis('Static')
ops.analyze(1)
ops.loadConst('-time', 0.0)

# ── Curvature sweep ──
n_steps = 300
max_curv = 8e-5
d_curv = max_curv / n_steps

ops.timeSeries('Linear', 2)
ops.pattern('Plain', 2, 2)
ops.load(2, 0.0, 0.0, 1.0)

ops.integrator('DisplacementControl', 2, 3, d_curv)
ops.analysis('Static')

curv_class = [0.0]
mom_class  = [0.0]

for i in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        ops.algorithm('NewtonLineSearch', 0.8)
        ok = ops.analyze(1)
        ops.algorithm('Newton')
    if ok != 0:
        print(f"Analysis failed at step {i+1}/{n_steps}")
        break
    curv_class.append(ops.nodeDisp(2, 3))
    mom_class.append(ops.eleForce(1, 3))

curv_class = np.array(curv_class)
mom_class  = np.abs(np.array(mom_class))

print(f"\nClass approach — {len(curv_class)-1}/{n_steps} steps completed")
print(f"Peak moment: {mom_class.max()/1e6:.1f} kN·m")
ops.wipe()

In [ ]:
# ── Final comparison: All three approaches ──
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(curv_raw   * 1e6, moment_raw / 1e6, 'b-',  linewidth=2.0, label='Raw Gmsh')
ax.plot(curv_pg    * 1e6, moment_pg  / 1e6, 'r--', linewidth=2.0, label='pyGmsh')
ax.plot(curv_class * 1e6, mom_class  / 1e6, 'g-.', linewidth=2.5, label='RectangularColumnSection')

ax.set_xlabel('Curvature [×10⁻⁶ 1/mm]', fontsize=12)
ax.set_ylabel('Moment [kN·m]', fontsize=12)
ax.set_title('Moment-Curvature: Three Approaches Compared', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

# ── Numerical summary ──
print("="*65)
print("FINAL COMPARISON — PEAK MOMENTS")
print("="*65)
for name, curv, mom in [("Raw Gmsh",                curv_raw,   moment_raw),
                         ("pyGmsh",                  curv_pg,    moment_pg),
                         ("RectangularColumnSection", curv_class, mom_class)]:
    idx = np.argmax(mom)
    print(f"  {name:30s}  M_max = {mom[idx]/1e6:7.1f} kN·m  "
          f"@ phi = {curv[idx]*1e6:.2f} x10^-6 1/mm")
print("="*65)

---
### Takeaways

All three approaches produce **identical moment-curvature curves**, confirming that
the `RectangularColumnSection` class correctly replicates the manual workflow.

| Approach | Lines of user code | What it handles |
|---|---|---|
| Raw Gmsh + OpenSeesPy | ~100 | Full control, educational |
| pyGmsh + OpenSeesPy | ~60 | Cleaner API, same flexibility |
| `RectangularColumnSection` | ~3 | Geometry + mesh + materials + fibers |

For production models, `RectangularColumnSection.build(sec_tag)` injects the
complete fiber section directly into your OpenSeesPy model — ready for frame
analysis, pushover, or dynamic simulations.

---
## Part 6: 3D Viewer Integration

Export the fiber discretization to a VTK `.vtu` file and launch **pyGmshViewer**
to inspect the section in the 3D viewport. Each fiber becomes a point with scalar
fields for *area*, *region* (core / cover / rebar), and the *strain* computed at
peak curvature via plane-sections (ε = -κ·z).

In the viewer you can:
- Switch scalar field → `Strain_peak`, `Area`, or `Region`
- Use **Tools → Point Probe** (Ctrl+P) to sample strain at any fiber
- Use **Tools → Line Probe** (Ctrl+L) to get a strain profile across the depth

In [ ]:
# ── Build a point-cloud VTU file from the fibers ──
from pathlib import Path
import numpy as np
import sys
sys.path.insert(0, '../src')
from pyGmsh.VTKExport import write_vtu

VTK_VERTEX = 1  # one-node cell

# Stack all fibers (core + cover + rebar) into a single point set
all_y = np.concatenate([pg_core_y, pg_cover_y, rebar_y])
all_z = np.concatenate([pg_core_z, pg_cover_z, rebar_z])
all_A = np.concatenate([pg_core_A, pg_cover_A, rebar_A])

region = np.concatenate([
    np.zeros(len(pg_core_y),  dtype=float),   # 0 = core
    np.ones (len(pg_cover_y), dtype=float),   # 1 = cover
    np.full (len(rebar_y), 2.0),              # 2 = rebar
])

# Peak curvature from the pyGmsh run
kappa_peak = curv_pg[np.argmax(moment_pg)]
# Plane-sections (about the centroid, axial ε₀ ignored for visualisation)
strain_peak = -kappa_peak * all_z

# Place fibers in the XY plane of a 3D VTU (x→y_fiber, y→z_fiber, z→0)
N = all_y.size
points = np.column_stack([all_y, all_z, np.zeros(N)])
cells  = np.arange(N, dtype=np.int32).reshape(-1, 1)   # one vertex per cell

out_path = Path('rc_fiber_section.vtu').resolve()
write_vtu(
    out_path,
    points=points,
    cells=cells,
    vtk_cell_type=VTK_VERTEX,
    point_data={
        'Area':        all_A,
        'Region':      region,
        'Strain_peak': strain_peak,
        'z_coord':     all_z,
    },
)
print(f'Wrote {N} fibers → {out_path}')
print(f'  kappa_peak = {kappa_peak*1e6:.3f} ×10⁻⁶ 1/mm')
print(f'  strain range: [{strain_peak.min():.3e}, {strain_peak.max():.3e}]')

In [ ]:
# ── Launch the pyGmshViewer (non-blocking, opens in a new process) ──
# Requires: pip install -e ".[viewer]" from the repo root
from pyGmshViewer import show

show(out_path, blocking=False)
print('Viewer launched. In the viewer:')
print('  1. Select "rc_fiber_section" in the Model tree')
print('  2. Under Contour, pick the "Strain_peak" scalar field')
print('  3. Use Ctrl+P to probe individual fibers')